In [1]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

load_dotenv()

llm = ChatOpenAI(
    model="prism-ml/ternary-bonsai-2-27b",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    max_tokens=1000,
)

In [10]:
import os

print(os.getcwd())

c:\Users\Lenovo\Desktop\Agentic AI\GuardRail_RAG_project


In [2]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

dir_loader = DirectoryLoader(
    "data",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=True
)

documents = dir_loader.load()

print(len(documents))

100%|██████████| 14/14 [00:16<00:00,  1.21s/it]

451


In [3]:
documents

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-12-19T17:27:04+05:00', 'author': 'Ehtisham Ahmad', 'keywords': 'ADP', 'moddate': '2025-12-19T17:27:04+05:00', 'source': 'data\\Accounting_and_Disclosure-Policy-2025.pdf', 'total_pages': 36, 'page': 0, 'page_label': '1'}, page_content='ACCOUNTING & DISCLOSURE POLICY FINANCIAL CONTROL GROUP \n \n \nACCOUNTING & DISCLOSURE POLICY \n \nFINANCIAL CONTROL GROUP'),
 Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-12-19T17:27:04+05:00', 'author': 'Ehtisham Ahmad', 'keywords': 'ADP', 'moddate': '2025-12-19T17:27:04+05:00', 'source': 'data\\Accounting_and_Disclosure-Policy-2025.pdf', 'total_pages': 36, 'page': 1, 'page_label': '2'}, page_content='ACCOUNTING & DISCLOSURE POLICY FINANCIAL CONTROL GROUP \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n

In [4]:
type(documents[0])

langchain_core.documents.base.Document

In [5]:
### Text splitting get into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter


def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [6]:
chunks=split_documents(documents, chunk_size=1000, chunk_overlap=200)
chunks

Split 451 documents into 1278 chunks

Example chunk:
Content: ACCOUNTING & DISCLOSURE POLICY FINANCIAL CONTROL GROUP 
 
 
ACCOUNTING & DISCLOSURE POLICY 
 
FINANCIAL CONTROL GROUP...
Metadata: {'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-12-19T17:27:04+05:00', 'author': 'Ehtisham Ahmad', 'keywords': 'ADP', 'moddate': '2025-12-19T17:27:04+05:00', 'source': 'data\\Accounting_and_Disclosure-Policy-2025.pdf', 'total_pages': 36, 'page': 0, 'page_label': '1'}


[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-12-19T17:27:04+05:00', 'author': 'Ehtisham Ahmad', 'keywords': 'ADP', 'moddate': '2025-12-19T17:27:04+05:00', 'source': 'data\\Accounting_and_Disclosure-Policy-2025.pdf', 'total_pages': 36, 'page': 0, 'page_label': '1'}, page_content='ACCOUNTING & DISCLOSURE POLICY FINANCIAL CONTROL GROUP \n \n \nACCOUNTING & DISCLOSURE POLICY \n \nFINANCIAL CONTROL GROUP'),
 Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-12-19T17:27:04+05:00', 'author': 'Ehtisham Ahmad', 'keywords': 'ADP', 'moddate': '2025-12-19T17:27:04+05:00', 'source': 'data\\Accounting_and_Disclosure-Policy-2025.pdf', 'total_pages': 36, 'page': 1, 'page_label': '2'}, page_content='ACCOUNTING & DISCLOSURE POLICY FINANCIAL CONTROL GROUP \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n

In [ ]:
## Embedding And VectorStoreDB
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
from langchain.messages import SystemMessage, HumanMessage

system_msg = SystemMessage("""
You are a banking guide with expertise in banking policies, procedures and services.
Always provide concrete examples and explain your reasoning.
Be concise but thorough in your explanations.
""")

messages = [
    system_msg,
    HumanMessage("How do I withdraw money from my account?")
]
response = llm.invoke(messages)
print(response.content)



To withdraw money from your account, use one of these common methods:

1. **ATM withdrawal**
   - Go to an ATM.
   - Insert your debit card and enter your PIN.
   - Select “Withdraw” and choose an amount.
   - **Example:** At a bank ATM, you withdraw $100 in cash.

2. **Bank branch/teller withdrawal**
   - Visit a branch with your government-issued ID.
   - Ask a teller to withdraw cash.
   - **Example:** You bring your driver’s license and debit card and request a $200 cash withdrawal.

3. **Mobile app or online banking transfer**
   - Log in to your bank’s app or website.
   - Choose “Transfers” or “Withdraw.”
   - Select the source and destination accounts.
   - **Example:** You transfer $500 from your checking account to your savings account.

4. **Check or money order**
   - Ask your bank to write a check to yourself or pay a bill.
   - Some banks require you to be present or call ahead.
   - **Example:** You request a $300 check, which you can deposit or cash later.

Important 